# This is an experiment to try to associate description to the ardc parameter vocab

It serve as a record only, as this approach isn't the best as the parameter field (aka parameter found in dataset) should be extract from the dataset. Although the field name can varies from dataset, but train a network to associate the field name to ardc vocab make much more sense.

In [1]:
!python3 -m pip install transformers datasets pandas scikit-learn "elasticsearch>=8,<9" "elasticsearch-dsl>=8,<9"

In [2]:
# import modules
from elasticsearch import Elasticsearch
from getpass import getpass

# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#finding-your-cloud-id
ELASTIC_CLOUD_ID = "dev-discovery-index:YXAtc291dGhlYXN0LTIuYXdzLmZvdW5kLmlvOjQ0MyQ2Y2E3ZTMxZWZmZDA0YzQzOGIxNGYxMWE0OWRjMjExOSQyMGNkYzdkYjQwZjU0ZGJiODMwMjgwZGNiMTFiYzA5Zg=="

# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

es = Elasticsearch(
    cloud_id=ELASTIC_CLOUD_ID, api_key=ELASTIC_API_KEY, request_timeout=600
)

es.info()  # should return cluster info

ObjectApiResponse({'name': 'instance-0000000000', 'cluster_name': '6ca7e31effd04c438b14f11a49dc2119', 'cluster_uuid': 'v96KrBz7SWGwlUsDeiqbmQ', 'version': {'number': '8.13.3', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '617f7b76c4ebcb5a7f1e70d409a99c437c896aea', 'build_date': '2024-04-29T22:05:16.051731935Z', 'build_snapshot': False, 'lucene_version': '9.10.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'})

In [3]:
# Find record where there are categories to create a dataset for learning
resp = es.search(
    index="es-indexer-edge", 
    size="1500", 
    query={
        "constant_score": {
          "filter": {
            "exists": {
              "field": "summaries.discovery_categories"
            }
          }
        }
    }
)
print("Got %d Hits:" % resp['hits']['total']['value'])
for hit in resp['hits']['hits']:
    print("%(description)s %(categories)s" % 
          {
              "description": hit["_source"]["description"],
              "categories": hit["_source"]["summaries"]["discovery_categories"]
          }
    )

Got 2234 Hits:
This dataset contains the processed Conductivity-Temperature-Depth (CTD) data collected on Mesocat voyage MC200306, processed by SRFME staff and is archived within the CSIRO Marine and Atmospheric Research Data Centre in Floreat. This dataset is part of the SRFME research program. ['water pressure', 'temperature', 'salinity']
This record contains data collected on the charter vessel RV Lady Basten between 23 and 28 August 2005 in the Great Barrier Reef. The data can be used for Ocean Colour sensor validation. Parameters measured are the concentration of chlorophyll and carotenoid pigments and retrieved chlorophyll a estimate, the absorption coefficient for dissolved (aCDOM) particulate (a/p) and detrital or non-algal (a/d) components of the water column. 20 stations were sampled. The data was used primarily to validate ocean colour sensors MERIS, MODIS and SeaWIFs and the SST sensor AATSR. Samples were collected and analysed for pigments, total suspended solids (TSS) and

In [4]:
# Create dataset for training
data = {
    "features" : [hit["_source"]["description"] for hit in resp['hits']['hits']],
    "labels": [hit["_source"]["summaries"]["discovery_categories"] for hit in resp['hits']['hits']]
}

print(data)

{'features': ['This dataset contains the processed Conductivity-Temperature-Depth (CTD) data collected on Mesocat voyage MC200306, processed by SRFME staff and is archived within the CSIRO Marine and Atmospheric Research Data Centre in Floreat. This dataset is part of the SRFME research program.', 'This record contains data collected on the charter vessel RV Lady Basten between 23 and 28 August 2005 in the Great Barrier Reef. The data can be used for Ocean Colour sensor validation. Parameters measured are the concentration of chlorophyll and carotenoid pigments and retrieved chlorophyll a estimate, the absorption coefficient for dissolved (aCDOM) particulate (a/p) and detrital or non-algal (a/d) components of the water column. 20 stations were sampled. The data was used primarily to validate ocean colour sensors MERIS, MODIS and SeaWIFs and the SST sensor AATSR. Samples were collected and analysed for pigments, total suspended solids (TSS) and  dissolved and particulate absorption coef

In [5]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

# Convert to DataFrame
df = pd.DataFrame(data)

# Vectorize the features (for example, using TF-IDF for text data)
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['features'])

# Binarize the labels
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['labels'])

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier

# Initialize the classifier
base_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
multi_target_classifier = MultiOutputClassifier(base_classifier, n_jobs=-1)

# Train the model
multi_target_classifier.fit(X_train, y_train)

MultiOutputClassifier(estimator=RandomForestClassifier(random_state=42),
                      n_jobs=-1)

In [7]:
from sklearn.metrics import accuracy_score, f1_score

# Predict on the test set
y_pred = multi_target_classifier.predict(X_test)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='micro')

print(f'Accuracy: {accuracy}')
print(f'F1 Score: {f1}')

Accuracy: 0.9466666666666667
F1 Score: 0.9773226042428674


In [8]:
# Convert y_pred back to original labels
y_pred_labels = mlb.inverse_transform(y_pred)

# Print the first few predictions
for i in range(5):
    print(f"Predicted labels for test instance {i}: {y_pred_labels[i]}")


Predicted labels for test instance 0: ('salinity', 'temperature')
Predicted labels for test instance 1: ('current',)
Predicted labels for test instance 2: ('current',)
Predicted labels for test instance 3: ('bathymetry', 'density')
Predicted labels for test instance 4: ('salinity', 'temperature', 'water pressure')


In [9]:
# Save the trained network
import pickle

# Save the model to a file
pickle_file = "multi_target_classifier.pkl"
with open(pickle_file, 'wb') as file:
    pickle.dump(multi_target_classifier, file)

with open('tfidf_vectorizer.pkl', 'wb') as file:
    pickle.dump(vectorizer, file)

In [10]:
# Load dataset where discovery_categories is null

# Find record where there are categories to create a dataset for learning
resp = es.search(
    index="es-indexer-edge", 
    size="10", 
    query={
        "bool": {
          "must_not": {
            "exists": {
              "field": "summaries.discovery_categories"
            }
          }
        }
    }
)

print("Got %d Hits:" % resp['hits']['total']['value'])
for hit in resp['hits']['hits']:
    print("%(description)s %(categories)s" % 
          {
              "description": hit["_source"]["description"],
              "categories": hit["_source"]["summaries"]["discovery_categories"]
          }
    )

Got 2275 Hits:
This record is an overview entry for biological data collected on Soela cruise SO 1/85. This cruise took place in Tasmanian waters and the eastern Bass Strait, under the leadership of T. Kenchington and Jock Young. Biological data collected on this cruise include demersal fish samples from trawling. Blue grenadier was the dominant species by weight. Morphometric and electrophoretic samples of blue grenadier. Juvenile blue grenadier samples. Pelagic samples from trawling included large catches of the lantern fish Lampanyctodes hectoris. Samples from longlining included small numbers of Brama brama, Rexea solandri, blue shark (Prionace glauca) and two Mako sharks (isurus oxyrhynchus). Ichthyoplankton and phytoplankton samples. Bottom fish samples. Seabird counts throughout the cruise, with several unusual species being observed. A large volume of cephalopod and other invertebrate samples (for National Museum of Victoria). Morphometric and electrophoretic samples of Helicol

In [11]:
# Typical Scenario with TF-IDF Vectorizer
#
# In the example provided earlier, we used TfidfVectorizer from scikit-learn for text vectorization. Here's how it handles new words:
#
#    Training Phase:
#        The TfidfVectorizer builds a vocabulary from the training data.
#        Each word in the training data is assigned a unique index in the feature space.
#
#    Prediction Phase:
#        When you transform new text data using the TfidfVectorizer, it maps known words to their corresponding indices in the feature space.
#        New words (i.e., words not seen during the training phase) are ignored because they do not exist in the vocabulary built during training. The TfidfVectorizer does not add new words to the vocabulary during transformation.

real_data = {
    "ids": [hit["_source"]["id"] for hit in resp['hits']['hits']],
    "features" : [hit["_source"]["description"] for hit in resp['hits']['hits']],
}

real_df = pd.DataFrame(real_data)
real_X = vectorizer.transform(real_df['features'])

real_label = mlb.inverse_transform(multi_target_classifier.predict(real_X))

for index, item in enumerate(real_label):
    print(item, real_data['ids'][index], real_data['features'][index])

('salinity', 'temperature') fa910097-0e7b-4d68-bb7d-f9534e15d10c This record is an overview entry for biological data collected on Soela cruise SO 1/85. This cruise took place in Tasmanian waters and the eastern Bass Strait, under the leadership of T. Kenchington and Jock Young. Biological data collected on this cruise include demersal fish samples from trawling. Blue grenadier was the dominant species by weight. Morphometric and electrophoretic samples of blue grenadier. Juvenile blue grenadier samples. Pelagic samples from trawling included large catches of the lantern fish Lampanyctodes hectoris. Samples from longlining included small numbers of Brama brama, Rexea solandri, blue shark (Prionace glauca) and two Mako sharks (isurus oxyrhynchus). Ichthyoplankton and phytoplankton samples. Bottom fish samples. Seabird counts throughout the cruise, with several unusual species being observed. A large volume of cephalopod and other invertebrate samples (for National Museum of Victoria). M